Carico Dataset


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!git clone https://github.com/Marco-2298/basketball-tracking-thesis.git
%cd basketball-tracking-thesis

fatal: destination path 'basketball-tracking-thesis' already exists and is not an empty directory.
/content/basketball-tracking-thesis


In [3]:
from pathlib import Path

DATASET_ROOT = Path(
    "/content/drive/MyDrive/basketball-thesis/datasets/sportsmot_coco"
)

print("Train esiste:", (DATASET_ROOT / "train").exists())
print("Valid esiste:", (DATASET_ROOT / "valid").exists())

Train esiste: True
Valid esiste: True


In [4]:
!ls -lh /content/drive/MyDrive/basketball-thesis/datasets

total 5.8G
-rw------- 1 root root 5.8G Sep 21 15:56 sportsmot_basketball_coco.zip
drwx------ 4 root root 4.0K Sep 21 16:04 sportsmot_coco


In [5]:
!rm -rf /content/data/sportsmot_coco


In [6]:
!unzip -q /content/drive/MyDrive/basketball-thesis/datasets/sportsmot_basketball_coco.zip -d /content/data

In [7]:
!find /content/data -maxdepth 4 -type d | head -50

/content/data
/content/data/sportsmot_coco
/content/data/sportsmot_coco/train
/content/data/sportsmot_coco/valid


In [8]:
from pathlib import Path

DATASET_ROOT = Path("/content/data/sportsmot_coco")

print("Train:", (DATASET_ROOT / "train").exists())
print("Valid:", (DATASET_ROOT / "valid").exists())

print("Train images:", len(list((DATASET_ROOT / "train").glob("*.jpg"))))
print("Valid images:", len(list((DATASET_ROOT / "valid").glob("*.jpg"))))

Train: True
Valid: True
Train images: 12384
Valid images: 12557


In [9]:
import torch

print("CUDA disponibile:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Nessuna")

CUDA disponibile: True
GPU: Tesla T4


In [10]:
!pip install -q rfdetr==1.10.1

*Cartella per i checkpoint training*

In [11]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/basketball-thesis")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "rfdetr_nano_smoke_test"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(CHECKPOINT_DIR)

/content/drive/MyDrive/basketball-thesis/checkpoints/rfdetr_nano_smoke_test


In [12]:
!pip install -q "rfdetr[train,loggers]==1.10.1"

In [14]:
import torch, sys

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4


In [15]:
!pip install -q nvidia-cuda-nvrtc

In [16]:
!find /usr/local/lib/python3.13/dist-packages/nvidia -name "libnvrtc-builtins.so.13*" -print

/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0


In [17]:
import os
import site

site_packages = site.getsitepackages()[0]
cu13_lib = f"{site_packages}/nvidia/cu13/lib"

os.environ["LD_LIBRARY_PATH"] = (
    cu13_lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
)

print(os.environ["LD_LIBRARY_PATH"])

/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib:/usr/local/lib/python3.13/dist-packages/cv2/../../lib64:/usr/lib64-nvidia


In [21]:
!find /usr/local/lib/python3.13/dist-packages -name "libnvrtc-builtins.so*" -print

/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12.9


In [22]:
!find /usr/local/cuda* -name "libnvrtc-builtins.so*" -print 2>/dev/null

/usr/local/cuda-13.3/targets/x86_64-linux/lib/libnvrtc-builtins.so
/usr/local/cuda-13.3/targets/x86_64-linux/lib/libnvrtc-builtins.so.13.3
/usr/local/cuda-13.3/targets/x86_64-linux/lib/libnvrtc-builtins.so.13.3.33


In [24]:
!find /usr/local/lib/python3.13/dist-packages -name "libnvrtc-builtins.so*" -print

/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0
/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc-builtins.so.12.9


In [25]:
import ctypes

ctypes.CDLL("/usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0")
print("NVRTC CUDA 13 OK")

NVRTC CUDA 13 OK


In [28]:
from rfdetr import RFDETRNano

DATASET_ROOT = "/content/data/sportsmot_coco"

model = RFDETRNano()

model.train(
    dataset_dir=DATASET_ROOT,
    epochs=1,
    batch_size="auto",
    lr=1e-4,
    output_dir=str(CHECKPOINT_DIR),
    progress_bar="tqdm",
)

[2026-09-23 17:19:27] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-09-23 17:19:27] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-23 17:19:27] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-23 17:19:29] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-09-23 17:19:33] [WARNING] rf-detr - [auto-batch] 1/465 trainable params received no gradient from the synthetic probe batch; their optimizer state is not modeled, so the returned batch size may be too permissive.


[2026-09-23 17:19:43] [INFO] rf-detr - [auto-batch] Applied EMA headroom (0.70): safe_micro_batch=4
[2026-09-23 17:19:43] [INFO] rf-detr - [auto-batch] device=Tesla T4 world_size=1 segmentation=False probe_resolution=544 max_targets_per_image=100
[2026-09-23 17:19:43] [INFO] rf-detr - [auto-batch] safe_micro_batch=4 grad_accum_steps=4 effective_batch_per_device=16 global_effective_batch=16
[2026-09-23 17:19:43] [INFO] rf-detr - [auto-batch] This probe estimates train-step-safe micro-batch only.
[2026-09-23 17:19:43] [INFO] rf-detr - [auto-batch] Validation/test (especially segmentation mask eval) may require more memory.
[2026-09-23 17:19:43] [INFO] rf-detr - [auto-batch] resolved train config: batch_size=4 grad_accum_steps=4 effective_batch_size=16


...

[2026-09-23 17:48:14] [INFO] rf-detr - Skipping validation-loss computation: compute_val_loss='auto' found no scheduler or callback monitoring 'val/loss', so no 'val/loss' metric is logged this run. Set compute_val_loss=True to log it every validation epoch (for example to read the curve from metrics.csv, TensorBoard, or Weights & Biases).


Output()

[2026-09-23 17:55:17] [INFO] rf-detr - Best EMA metric improved to 0.7000 (epoch 0)


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=1` reached.


[2026-09-23 17:55:28] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.7000)


## Risultati smoke test

Modello: RF-DETR Nano
Dataset: SportsMOT Basketball
Training: 1 epoca
GPU: NVIDIA Tesla T4

Risultati validation:
- mAP@50:95 = 0.7000
- mAP@50 = 0.9835
- mAP@75 = 0.8037
- Precision = 0.9755
- Recall = 0.9581
- F1 = 0.9667
- mAR@500 = 0.7916

Durata training: circa 35 minuti.

Il test ha verificato correttamente:
- caricamento dataset;
- training;
- validation;
- calcolo metriche;
- salvataggio checkpoint EMA.
